In [2]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('dataset\\train.csv')

In [3]:
df.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [4]:
df.isnull().sum()

Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64

In [5]:
df['RoadType'].value_counts()

RoadType
Residential    69230
Street          3909
Highway         3560
Name: count, dtype: int64

In [6]:
df['Weather'].value_counts()

Weather
Sunny    27717
Rainy    20824
Foggy    20243
Snowy     7718
Name: count, dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.groupby('Weather')['Temperature'].agg(['count', 'median', 'std'])

,count,median,std
Weather,,,
Foggy,19584,16.475623,1.434400
Rainy,20153,11.141699,1.969005
Snowy,7478,4.403676,2.995841
Sunny,26818,23.049061,4.031399


In [11]:

df.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,Temperature_missing,Weather_missing
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,Unknown,1,1
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,0,0
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,0,0
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,11.141699,Rainy,0,0
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,0,0


In [12]:
df.isnull().sum()

Index                    0
geohash                  0
day                      0
timestamp                0
demand                   0
RoadType               600
NumberofLanes            0
LargeVehicles            0
Landmarks                0
Temperature            797
Weather                  0
Temperature_missing      0
Weather_missing          0
dtype: int64

In [13]:
df['RoadType'] = df['RoadType'].fillna(df['RoadType'].mode()[0])

In [14]:
df['RoadType'].value_counts()

RoadType
Residential    69830
Street          3909
Highway         3560
Name: count, dtype: int64

In [18]:
df.isnull().sum()

Index                  0
geohash                0
day                    0
timestamp              0
demand                 0
RoadType               0
NumberofLanes          0
LargeVehicles          0
Landmarks              0
Temperature            0
Weather                0
Temperature_missing    0
Weather_missing        0
Both_missing           0
dtype: int64

In [26]:
df['geohash'].nunique()

1249

In [7]:
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

# =============================================================================
# 1. Load Data
# =============================================================================
train_df = pd.read_csv('dataset\\train.csv')
test_df = pd.read_csv('dataset\\test.csv')

if 'Index' in train_df.columns:
    train_df = train_df.drop(columns=['Index'])

test_index = test_df['Index'].copy() if 'Index' in test_df.columns else pd.Series(np.arange(len(test_df)))

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Target mean: {train_df['demand'].mean():.6f}, std: {train_df['demand'].std():.6f}")

# =============================================================================
# 2. Feature Engineering
# =============================================================================
def extract_time_features(df):
    df = df.copy()
    time_split = df['timestamp'].astype(str).str.split(':', expand=True)
    df['hour'] = time_split[0].astype(int)
    df['minute'] = time_split[1].astype(int)
    df['total_minutes'] = df['hour'] * 60 + df['minute']
    df['time_slot'] = df['total_minutes'] // 15
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['slot_sin'] = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['slot_cos'] = np.cos(2 * np.pi * df['time_slot'] / 96)
    df['is_morning_rush'] = ((df['hour'] >= 7) & (df['hour'] <= 10)).astype(int)
    df['is_evening_rush'] = ((df['hour'] >= 16) & (df['hour'] <= 19)).astype(int)
    df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
    df['is_peak'] = ((df['hour'] >= 7) & (df['hour'] <= 13)).astype(int)
    return df


def encode_categoricals(df):
    df = df.copy()
    df['Weather_missing'] = df['Weather'].isna().astype(int)
    df['Temperature_missing'] = df['Temperature'].isna().astype(int)
    df['Both_missing'] = (df['Weather'].isna() & df['Temperature'].isna()).astype(int)
    df['Weather'] = df['Weather'].fillna('Unknown')
    df['RoadType'] = df['RoadType'].fillna(df['RoadType'].mode()[0])
    df['LargeVehicles'] = df['LargeVehicles'].fillna(df['LargeVehicles'].mode()[0])
    df['Landmarks'] = df['Landmarks'].fillna(df['Landmarks'].mode()[0])
    global_temp_median = df['Temperature'].median()
    weather_temp_medians = df.groupby('Weather')['Temperature'].median()
    df['Temperature'] = df.apply(
        lambda row: weather_temp_medians.get(row['Weather'], global_temp_median)
        if pd.isna(row['Temperature']) else row['Temperature'],
        axis=1
    )
    road_map = {'Residential': 0, 'Street': 1, 'Highway': 2}
    df['RoadType_enc'] = df['RoadType'].map(road_map).fillna(0)
    df['is_highway'] = (df['RoadType'] == 'Highway').astype(int)
    df['is_street'] = (df['RoadType'] == 'Street').astype(int)
    df['LargeVehicles_enc'] = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['Landmarks_enc'] = (df['Landmarks'] == 'Yes').astype(int)
    weather_map = {'Sunny': 0, 'Rainy': 1, 'Foggy': 2, 'Snowy': 3}
    df['Weather_enc'] = df['Weather'].map(weather_map).fillna(-1)
    for w in ['Sunny', 'Rainy', 'Foggy', 'Snowy']:
        df[f'weather_{w.lower()}'] = (df['Weather'] == w).astype(int)
    return df


def decode_geohash(gh):
    if pd.isna(gh):
        return np.nan, np.nan
    base32 = '0123456789bcdefghjkmnpqrstuvwxyz'
    bits = [16, 8, 4, 2, 1]
    decode_map = {c: i for i, c in enumerate(base32)}
    gh = str(gh).strip().lower()
    lat_range = [-90.0, 90.0]
    lon_range = [-180.0, 180.0]
    even = True
    for ch in gh:
        cd = decode_map.get(ch)
        if cd is None:
            return np.nan, np.nan
        for mask in bits:
            if even:
                mid = (lon_range[0] + lon_range[1]) / 2
                if cd & mask:
                    lon_range[0] = mid
                else:
                    lon_range[1] = mid
            else:
                mid = (lat_range[0] + lat_range[1]) / 2
                if cd & mask:
                    lat_range[0] = mid
                else:
                    lat_range[1] = mid
            even = not even
    return (lat_range[0] + lat_range[1]) / 2, (lon_range[0] + lon_range[1]) / 2


def build_geohash_features(train, test, target_col='demand'):
    train = train.copy()
    test = test.copy()
    global_mean = train[target_col].mean()
    geo_stats = train.groupby('geohash')[target_col].agg(['mean', 'median', 'std', 'count', 'min', 'max']).reset_index()
    geo_stats.columns = ['geohash', 'geo_demand_mean', 'geo_demand_median', 'geo_demand_std', 'geo_count', 'geo_demand_min', 'geo_demand_max']
    smoothing = 10
    geo_stats['geo_target_enc'] = (
        (geo_stats['geo_count'] * geo_stats['geo_demand_mean'] + smoothing * global_mean) /
        (geo_stats['geo_count'] + smoothing)
    )
    geo_stats['geo_demand_range'] = geo_stats['geo_demand_max'] - geo_stats['geo_demand_min']
    geo_stats['geo_demand_std'] = geo_stats['geo_demand_std'].fillna(0)
    train = train.merge(geo_stats, on='geohash', how='left')
    test = test.merge(geo_stats, on='geohash', how='left')
    for col in ['geo_demand_mean', 'geo_demand_median', 'geo_target_enc']:
        test[col] = test[col].fillna(global_mean)
    test['geo_demand_std'] = test['geo_demand_std'].fillna(0)
    test['geo_count'] = test['geo_count'].fillna(0)
    test['geo_demand_min'] = test['geo_demand_min'].fillna(global_mean)
    test['geo_demand_max'] = test['geo_demand_max'].fillna(global_mean)
    test['geo_demand_range'] = test['geo_demand_range'].fillna(0)
    train['hour_tmp'] = train['timestamp'].astype(str).str.split(':').str[0].astype(int)
    test['hour_tmp'] = test['timestamp'].astype(str).str.split(':').str[0].astype(int)
    geo_hour = train.groupby(['geohash', 'hour_tmp'])[target_col].agg(['mean', 'count']).reset_index()
    geo_hour.columns = ['geohash', 'hour_tmp', 'geo_hour_demand_mean', 'geo_hour_count']
    geo_hour['geo_hour_target_enc'] = (
        (geo_hour['geo_hour_count'] * geo_hour['geo_hour_demand_mean'] + smoothing * global_mean) /
        (geo_hour['geo_hour_count'] + smoothing)
    )
    train = train.merge(geo_hour[['geohash', 'hour_tmp', 'geo_hour_target_enc']], on=['geohash', 'hour_tmp'], how='left')
    test = test.merge(geo_hour[['geohash', 'hour_tmp', 'geo_hour_target_enc']], on=['geohash', 'hour_tmp'], how='left')
    train['geo_hour_target_enc'] = train['geo_hour_target_enc'].fillna(train['geo_target_enc'])
    test['geo_hour_target_enc'] = test['geo_hour_target_enc'].fillna(test['geo_target_enc'])
    geo_road = train.groupby(['geohash', 'RoadType'])[target_col].mean().reset_index()
    geo_road.columns = ['geohash', 'RoadType', 'geo_road_demand']
    train = train.merge(geo_road, on=['geohash', 'RoadType'], how='left')
    test = test.merge(geo_road, on=['geohash', 'RoadType'], how='left')
    test['geo_road_demand'] = test['geo_road_demand'].fillna(global_mean)
    train = train.drop(columns=['hour_tmp'])
    test = test.drop(columns=['hour_tmp'])
    return train, test


def add_interaction_features(df):
    df = df.copy()
    df['lanes_highway'] = df['NumberofLanes'] * df['is_highway']
    df['lanes_street'] = df['NumberofLanes'] * df['is_street']
    df['high_capacity'] = ((df['NumberofLanes'] >= 4) | (df['is_highway'] == 1)).astype(int)
    df['temp_is_highway'] = df['Temperature'] * df['is_highway']
    df['geo_mean_x_hour'] = df['geo_demand_mean'] * df['hour_sin']
    df['lanes_large_vehicles'] = df['NumberofLanes'] * df['LargeVehicles_enc']
    df['geo_mean_vs_median'] = df['geo_demand_mean'] / (df['geo_demand_median'] + 1e-8)
    return df


def full_pipeline(train_df, test_df):
    train = extract_time_features(train_df)
    test = extract_time_features(test_df)
    train = encode_categoricals(train)
    test = encode_categoricals(test)
    decoded_train = train['geohash'].apply(decode_geohash)
    decoded_test = test['geohash'].apply(decode_geohash)
    train['geo_lat'] = decoded_train.str[0]
    train['geo_lon'] = decoded_train.str[1]
    test['geo_lat'] = decoded_test.str[0]
    test['geo_lon'] = decoded_test.str[1]
    train, test = build_geohash_features(train, test)
    train = add_interaction_features(train)
    test = add_interaction_features(test)
    drop_cols = ['timestamp', 'geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'Index']
    y = train['demand']
    train = train.drop(columns=['demand'] + [c for c in drop_cols if c in train.columns])
    test = test.drop(columns=[c for c in drop_cols if c in test.columns])
    train = train.fillna(train.median(numeric_only=True))
    test = test.fillna(train.median(numeric_only=True))
    train = train.replace([np.inf, -np.inf], np.nan).fillna(train.median(numeric_only=True))
    test = test.replace([np.inf, -np.inf], np.nan).fillna(train.median(numeric_only=True))
    common_cols = [c for c in train.columns if c in test.columns]
    return train[common_cols], y, test[common_cols]


# =============================================================================
# 3. Build Features
# =============================================================================
X_train_full, y_full, X_test_full = full_pipeline(train_df, test_df)
feature_cols = X_train_full.columns.tolist()

print(f"\nFeatures: {len(feature_cols)}")
print(f"Feature names: {feature_cols}")
print(f"Training feature shape: {X_train_full.shape}")
print(f"Test feature shape: {X_test_full.shape}")

Train shape: (77299, 10)
Test shape: (41778, 10)
Target mean: 0.093942, std: 0.142191

Features: 47
Feature names: ['day', 'NumberofLanes', 'Temperature', 'hour', 'minute', 'total_minutes', 'time_slot', 'hour_sin', 'hour_cos', 'slot_sin', 'slot_cos', 'is_morning_rush', 'is_evening_rush', 'is_night', 'is_peak', 'Weather_missing', 'Temperature_missing', 'Both_missing', 'RoadType_enc', 'is_highway', 'is_street', 'LargeVehicles_enc', 'Landmarks_enc', 'Weather_enc', 'weather_sunny', 'weather_rainy', 'weather_foggy', 'weather_snowy', 'geo_lat', 'geo_lon', 'geo_demand_mean', 'geo_demand_median', 'geo_demand_std', 'geo_count', 'geo_demand_min', 'geo_demand_max', 'geo_target_enc', 'geo_demand_range', 'geo_hour_target_enc', 'geo_road_demand', 'lanes_highway', 'lanes_street', 'high_capacity', 'temp_is_highway', 'geo_mean_x_hour', 'lanes_large_vehicles', 'geo_mean_vs_median']
Training feature shape: (77299, 47)
Test feature shape: (41778, 47)


In [10]:
# =============================================================================
# 4. Model Training with K-Fold CV + Tuned Ensemble
# =============================================================================
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Keep parameter keys unchanged, but try a small set of diverse values.
lgb_param_options = [
    {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'learning_rate': 0.03,
        'num_leaves': 127,
        'max_depth': -1,
        'min_child_samples': 20,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'n_estimators': 2000,
        'random_state': 42,
        'verbosity': -1,
    },
    {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'learning_rate': 0.01,
        'num_leaves': 127,
        'max_depth': -1,
        'min_child_samples': 20,
        'feature_fraction': 0.85,
        'bagging_fraction': 0.85,
        'bagging_freq': 5,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'n_estimators': 3000,
        'random_state': 42,
        'verbosity': -1,
    },
]

xgb_param_options = [
    {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'learning_rate': 0.03,
        'max_depth': 8,
        'min_child_weight': 10,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'n_estimators': 2000,
        'random_state': 42,
        'tree_method': 'hist',
        'verbosity': 0,
    },
    {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'learning_rate': 0.01,
        'max_depth': 7,
        'min_child_weight': 8,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'n_estimators': 3000,
        'random_state': 42,
        'tree_method': 'hist',
        'verbosity': 0,
    },
]

cat_param_options = [
    {
        'iterations': 1500,
        'learning_rate': 0.03,
        'depth': 8,
        'l2_leaf_reg': 3.0,
        'loss_function': 'RMSE',
        'random_seed': 42,
        'verbose': False,
        'allow_writing_files': False,
        'od_type': 'Iter',
        'od_wait': 100,
    },
    {
        'iterations': 2200,
        'learning_rate': 0.02,
        'depth': 8,
        'l2_leaf_reg': 4.0,
        'loss_function': 'RMSE',
        'random_seed': 42,
        'verbose': False,
        'allow_writing_files': False,
        'od_type': 'Iter',
        'od_wait': 100,
    },
]

oof_lgb = np.zeros(len(X_train_full))
oof_xgb = np.zeros(len(X_train_full))
oof_cat = np.zeros(len(X_train_full))
test_lgb = np.zeros(len(X_test_full))
test_xgb = np.zeros(len(X_test_full))
test_cat = np.zeros(len(X_test_full))

print(f"\n{'='*60}")
print(f"Training with {N_FOLDS}-Fold Cross Validation")
print(f"Parameter options per model: LGB={len(lgb_param_options)}, XGB={len(xgb_param_options)}, CAT={len(cat_param_options)}")
print(f"{'='*60}")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_full)):
    print(f"\n--- Fold {fold + 1}/{N_FOLDS} ---")
    X_tr, X_val = X_train_full.iloc[train_idx], X_train_full.iloc[val_idx]
    y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]

    best_lgb_fold_r2 = -np.inf
    best_lgb_val_pred = None
    best_lgb_test_pred = None
    best_lgb_idx = -1

    for idx, params in enumerate(lgb_param_options):
        lgb_model = LGBMRegressor(**params)
        lgb_model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )
        val_pred = lgb_model.predict(X_val)
        fold_r2 = r2_score(y_val, val_pred)
        if fold_r2 > best_lgb_fold_r2:
            best_lgb_fold_r2 = fold_r2
            best_lgb_val_pred = val_pred
            best_lgb_test_pred = lgb_model.predict(X_test_full)
            best_lgb_idx = idx

    oof_lgb[val_idx] = best_lgb_val_pred
    test_lgb += best_lgb_test_pred / N_FOLDS
    print(f"  LGB best option: {best_lgb_idx + 1}, R2: {best_lgb_fold_r2:.6f}")

    best_xgb_fold_r2 = -np.inf
    best_xgb_val_pred = None
    best_xgb_test_pred = None
    best_xgb_idx = -1

    for idx, params in enumerate(xgb_param_options):
        xgb_model = xgb.XGBRegressor(**params)
        xgb_model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
        val_pred = xgb_model.predict(X_val)
        fold_r2 = r2_score(y_val, val_pred)
        if fold_r2 > best_xgb_fold_r2:
            best_xgb_fold_r2 = fold_r2
            best_xgb_val_pred = val_pred
            best_xgb_test_pred = xgb_model.predict(X_test_full)
            best_xgb_idx = idx

    oof_xgb[val_idx] = best_xgb_val_pred
    test_xgb += best_xgb_test_pred / N_FOLDS
    print(f"  XGB best option: {best_xgb_idx + 1}, R2: {best_xgb_fold_r2:.6f}")

    best_cat_fold_r2 = -np.inf
    best_cat_val_pred = None
    best_cat_test_pred = None
    best_cat_idx = -1

    for idx, params in enumerate(cat_param_options):
        cat_model = CatBoostRegressor(**params)
        cat_model.fit(
            X_tr,
            y_tr,
            eval_set=(X_val, y_val),
            use_best_model=True,
        )
        val_pred = cat_model.predict(X_val)
        fold_r2 = r2_score(y_val, val_pred)
        if fold_r2 > best_cat_fold_r2:
            best_cat_fold_r2 = fold_r2
            best_cat_val_pred = val_pred
            best_cat_test_pred = cat_model.predict(X_test_full)
            best_cat_idx = idx

    oof_cat[val_idx] = best_cat_val_pred
    test_cat += best_cat_test_pred / N_FOLDS
    print(f"  CAT best option: {best_cat_idx + 1}, R2: {best_cat_fold_r2:.6f}")

lgb_oof_r2 = r2_score(y_full, oof_lgb)
xgb_oof_r2 = r2_score(y_full, oof_xgb)
cat_oof_r2 = r2_score(y_full, oof_cat)

print(f"\n{'='*60}")
print(f"Overall OOF R2 - LGB: {lgb_oof_r2:.6f}")
print(f"Overall OOF R2 - XGB: {xgb_oof_r2:.6f}")
print(f"Overall OOF R2 - CAT: {cat_oof_r2:.6f}")

best_r2 = -1
best_weights = (0.34, 0.33, 0.33)

for w_lgb in np.arange(0.0, 1.01, 0.05):
    for w_xgb in np.arange(0.0, 1.01 - w_lgb, 0.05):
        w_cat = round(1.0 - w_lgb - w_xgb, 10)
        if w_cat < 0:
            continue
        blend = w_lgb * oof_lgb + w_xgb * oof_xgb + w_cat * oof_cat
        r2 = r2_score(y_full, blend)
        if r2 > best_r2:
            best_r2 = r2
            best_weights = (w_lgb, w_xgb, w_cat)

print(f"\nBest ensemble weights [LGB, XGB, CAT]: {best_weights}")
print(f"Best ensemble OOF R2: {best_r2:.6f} | Hackathon Score: {max(0, 100 * best_r2):.4f}")

final_test_pred = best_weights[0] * test_lgb + best_weights[1] * test_xgb + best_weights[2] * test_cat
final_test_pred = np.clip(final_test_pred, 0, None)

print('Final test prediction preview:', final_test_pred[:5])


Training with 5-Fold Cross Validation
Parameter options per model: LGB=2, XGB=2, CAT=2

--- Fold 1/5 ---
  LGB best option: 2, R2: 0.972178
  XGB best option: 2, R2: 0.972565
  CAT best option: 1, R2: 0.973307

--- Fold 2/5 ---
  LGB best option: 2, R2: 0.973961
  XGB best option: 2, R2: 0.973562
  CAT best option: 2, R2: 0.974079

--- Fold 3/5 ---
  LGB best option: 2, R2: 0.973655
  XGB best option: 2, R2: 0.974156
  CAT best option: 1, R2: 0.974640

--- Fold 4/5 ---
  LGB best option: 2, R2: 0.970071
  XGB best option: 2, R2: 0.969683
  CAT best option: 2, R2: 0.970721

--- Fold 5/5 ---
  LGB best option: 2, R2: 0.974700
  XGB best option: 2, R2: 0.974968
  CAT best option: 1, R2: 0.975677

Overall OOF R2 - LGB: 0.972956
Overall OOF R2 - XGB: 0.973031
Overall OOF R2 - CAT: 0.973724

Best ensemble weights [LGB, XGB, CAT]: (np.float64(0.35000000000000003), np.float64(0.0), np.float64(0.65))
Best ensemble OOF R2: 0.974015 | Hackathon Score: 97.4015
Final test prediction preview: [0.04

In [5]:
test_df = pd.read_csv('dataset\\test.csv')

In [11]:
submission = pd.DataFrame({
    'Index': test_index.astype(int),
    'demand': final_test_pred
})

submission.to_csv('submission_ensemble_tuned.csv', index=False)
print('Saved: submission_ensemble_tuned.csv')
print(submission.head())

Saved: submission_ensemble_tuned.csv
   Index    demand
0      0  0.049502
1      1  0.031135
2      2  0.036191
3      3  0.077054
4      4  0.055254
